# 01 — Data Exploration

# SIR-EEGNet | EEG-based Alzheimer's Disease Classification

In [2]:
# %% 
# Install + download dataset (derivatives)

import subprocess
import os

subprocess.run(['pip', 'install', 'awscli', '-q'], check=True)

os.makedirs('/kaggle/working/ds004504/derivatives', exist_ok=True)

result = subprocess.run([
    'aws', 's3', 'sync',
    's3://openneuro.org/ds004504/derivatives/',
    '/kaggle/working/ds004504/derivatives/',
    '--no-sign-request',
    '--region', 'us-east-1'
], capture_output=True, text=True)

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)

if result.returncode == 0:
    print("\nDownload complete.")
else:
    print("Error:", result.stderr[-1000:])



Download complete.


In [3]:
# %% 
# Download labels file

result2 = subprocess.run([
    'aws', 's3', 'cp',
    's3://openneuro.org/ds004504/participants.tsv',
    '/kaggle/working/ds004504/participants.tsv',
    '--no-sign-request',
    '--region', 'us-east-1'
], capture_output=True, text=True)

print(result2.stdout)

if result2.returncode == 0:
    print("participants.tsv downloaded.")
else:
    print("Error:", result2.stderr)

Completed 1.7 KiB/1.7 KiB (5.3 KiB/s) with 1 file(s) remaining
download: s3://openneuro.org/ds004504/participants.tsv to ../kaggle/working/ds004504/participants.tsv

participants.tsv downloaded.


In [4]:
# %% 
# Install dependencies

import subprocess, sys

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'], check=True)

pip('mne', 'awscli', 'scipy', 'matplotlib', 'seaborn', 'pandas', 'numpy')
print('Dependencies ready.')

Dependencies ready.


In [5]:
# %% 
# Paths + environment setup

import os
from pathlib import Path

DATA_PATH = Path("/kaggle/working/ds004504/")
EEG_PATH = DATA_PATH / "derivatives"
LABELS_PATH = DATA_PATH / "participants.tsv"

if os.path.exists('/content'):
    ENV, BASE = 'colab', Path('/content/sir-eegnet')
elif os.path.exists('/kaggle/working'):
    ENV, BASE = 'kaggle', Path('/kaggle/working/sir-eegnet')
else:
    ENV, BASE = 'local', Path.cwd().parent

DATA_DIR        = BASE / 'data'
DS_DIR          = DATA_DIR / 'ds004504'
DERIV_DIR       = EEG_PATH
PARTICIPANTS    = LABELS_PATH
RESULTS_DIR     = BASE / 'results'

for d in [DATA_DIR, DS_DIR, DERIV_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Env: {ENV} | Base: {BASE}')

Env: colab | Base: /content/sir-eegnet


In [6]:
# %% 
# Constants + config

import numpy as np
import pandas as pd
np.random.seed(42)

SFREQ          = 500.0
N_CHANNELS     = 19
EPOCH_DURATION = 4.0
EPOCH_OVERLAP  = 0.5
EPOCH_SAMPLES  = int(EPOCH_DURATION * SFREQ)

GROUP_MAP   = {'A': 0, 'F': 1, 'C': 2}
LABEL_NAMES = {0: 'AD', 1: 'FTD', 2: 'HC'}
COLORS      = {'AD': '#E74C3C', 'FTD': '#F39C12', 'HC': '#27AE60'}

FREQ_BANDS = {
    'delta': (0.5, 4.0),
    'theta': (4.0, 8.0),
    'alpha': (8.0, 13.0),
    'beta':  (13.0, 30.0),
    'gamma': (30.0, 45.0),
}

print('Constants set. Epoch samples:', EPOCH_SAMPLES)

Constants set. Epoch samples: 2000


In [7]:
# %% 
# Load participants + sanity check

participants = pd.read_csv(PARTICIPANTS, sep='\t')
participants['label'] = participants['Group'].map(GROUP_MAP)
participants = participants.dropna(subset=['label'])
participants['label'] = participants['label'].astype(int)

print(f'Total subjects: {len(participants)}')

print('\nClass distribution:')
for label, name in LABEL_NAMES.items():
    n = (participants['label'] == label).sum()
    print(f'  {name}: {n} subjects')

print('\nAge by group:')
print(participants.groupby('Group')['Age'].describe().round(1))

participants.head()

Total subjects: 88

Class distribution:
  AD: 36 subjects
  FTD: 23 subjects
  HC: 29 subjects

Age by group:
       count  mean  std   min   25%   50%   75%   max
Group                                                
A       36.0  66.4  7.9  49.0  61.0  67.0  71.2  79.0
C       29.0  67.9  5.4  57.0  64.0  67.0  71.0  78.0
F       23.0  63.7  8.2  44.0  60.5  64.0  70.5  78.0


,participant_id,Gender,Age,Group,MMSE,label
0,sub-001,F,57,A,16,0
1,sub-002,F,78,A,22,0
2,sub-003,M,70,A,14,0
3,sub-004,F,67,A,20,0
4,sub-005,M,70,A,22,0


In [8]:
# %% 
# File verification + sample inspection + epoching + helpers

missing, found = [], []
for _, row in participants.iterrows():
    sid = row['participant_id']
    f   = DERIV_DIR / sid / 'eeg' / f'{sid}_task-eyesclosed_eeg.set'
    (found if f.exists() else missing).append(sid)

print(f'Files found   : {len(found)}')
print(f'Files missing : {len(missing)}')

if missing:
    print('Missing:', missing)
else:
    print('All subject EEG files present.')


import mne
mne.set_log_level('WARNING')

SAMPLE = 'sub-001'
f = DERIV_DIR / SAMPLE / 'eeg' / f'{SAMPLE}_task-eyesclosed_eeg.set'
raw = mne.io.read_raw_eeglab(str(f), preload=True, verbose=False)

print('=== sub-001 (AD) EEG Info ===')
print(f'Sampling freq  : {raw.info["sfreq"]} Hz')
print(f'Channels       : {len(raw.ch_names)}')
print(f'Channel names  : {raw.ch_names}')
print(f'Duration       : {raw.times[-1]:.1f} s')
print(f'Data shape     : {raw.get_data().shape}')


def epoch_raw_signal(data, sfreq=500.0, duration=4.0, overlap=0.5):
    n_ch, n_t = data.shape
    ep_s   = int(duration * sfreq)
    step_s = int(ep_s * (1 - overlap))
    starts = range(0, n_t - ep_s + 1, step_s)
    return np.stack([data[:, s:s+ep_s] for s in starts], axis=0)

data   = raw.get_data()
epochs = epoch_raw_signal(data, SFREQ, EPOCH_DURATION, EPOCH_OVERLAP)

print(f'Raw data : {data.shape}')
print(f'Epochs   : {epochs.shape}')


from scipy.signal import welch
from scipy import stats

def compute_band_power(epoch, sfreq, freq_bands):
    nperseg = min(256, epoch.shape[1])
    freqs, psd = welch(epoch, fs=sfreq, nperseg=nperseg, axis=-1)
    res = freqs[1] - freqs[0]
    return {b: np.sum(psd[:, (freqs>=lo)&(freqs<=hi)], axis=-1)*res
            for b, (lo, hi) in freq_bands.items()}

def compute_dtabr_scalar(epoch, sfreq, freq_bands):
    bp  = compute_band_power(epoch, sfreq, freq_bands)
    num = bp['delta'] + bp['theta']
    den = bp['alpha'] + bp['beta'] + 1e-10
    return float(np.mean(num / den))

def subject_mean_psd(data, sfreq, nperseg=256):
    freqs, psd = welch(data, fs=sfreq, nperseg=nperseg, axis=-1)
    return freqs, psd.mean(axis=0)

print('Helper functions defined.')

Files found   : 88
Files missing : 0
All subject EEG files present.
=== sub-001 (AD) EEG Info ===
Sampling freq  : 500.0 Hz
Channels       : 19
Channel names  : ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz']
Duration       : 599.8 s
Data shape     : (19, 299900)
Raw data : (19, 299900)
Epochs   : (298, 19, 2000)
Helper functions defined.
